# Full Pipeline: V1 -> V6 (single file, any dataset)

```
full_pipeline_all_versions.py
-----------------------------------------------
ONE file containing the full V1 -> V6 modelling progression, faithfully
reproducing the methodology of your six original notebooks:

  V1  Raw baseline           dropna, label-encode, no scaling, no SMOTE
  V2  Imputed + scaled       split BEFORE impute, mean/median impute,
                             label-encode, MinMaxScaler (all fit on train only)
  V3  V2 + SMOTE             SMOTE nested inside the pipeline, refit per fold
  V4  V2 + feature selection Chi-square / RFE / RF-importance compared,
                             8 feature configs each run through SMOTE
  V5  V4(best config) tuned  RFE-Top-20 (V4's winning config) + SMOTE +
                             GridSearchCV hyperparameter tuning per model
                             (exhaustive search over every combination in
                             each model's param_grid, matching the
                             individual V5 notebook)
  V6  Stacking ensemble      RFE-Top-20 + SMOTE + Stacking(RF+SVM+NB -> LogReg)
                             using FIXED hyperparameters (the individual
                             V5 notebook's actual best_params_ for this
                             dataset, hardcoded -- not read dynamically
                             from whatever V5 produces this run)

DESIGN
- Config block at the top (DATA_PATH, TARGET_COL, COLS_TO_REMOVE,
  RANDOM_STATE) is the ONLY thing that should ever change between datasets/
  experiments -- exactly like your original per-version notebooks.
- The data is loaded ONCE. Each version function receives it and performs
  its own version-specific split/preprocessing internally (this
  deliberately mirrors your original notebooks: V1 splits AFTER dropna,
  V2-V6 split BEFORE imputation -- that difference is real methodology,
  not an inconsistency to "fix").
- No SHAP / figures here by design -- this file is for fast, comparable
  experiment results across datasets. Keep your original per-version
  notebooks for the full visual/SHAP analysis on a given dataset.

USAGE
  1. Prepare your dataset with prepare_dataset_for_pipeline.py first, so
     the target column is already named to match TARGET_COL below.
  2. Update DATA_PATH / TARGET_COL / COLS_TO_REMOVE for this experiment.
  3. Run: python full_pipeline_all_versions.py
  4. Read the final V1-V6 comparison table printed at the end.
```

## (Optional) Mount Google Drive
Run this if `DATA_PATH` below points to a file in Google Drive.

In [1]:
from google.colab import drive
drive.mount('/content/drive')


Mounted at /content/drive


## Install / Import

In [2]:
!pip install imbalanced-learn --quiet

import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd

from sklearn.preprocessing import LabelEncoder, MinMaxScaler, OrdinalEncoder
from sklearn.impute import SimpleImputer
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline as SkPipeline
from sklearn.model_selection import (
    train_test_split, StratifiedKFold, cross_val_score, GridSearchCV
)
from sklearn.tree import DecisionTreeClassifier
from sklearn.base import clone, BaseEstimator, TransformerMixin
from sklearn.ensemble import RandomForestClassifier, StackingClassifier
from sklearn.svm import SVC
from sklearn.neighbors import KNeighborsClassifier
from sklearn.naive_bayes import GaussianNB
from sklearn.linear_model import LogisticRegression
from sklearn.feature_selection import SelectKBest, chi2, RFE
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score

from imblearn.over_sampling import SMOTE
from imblearn.pipeline import Pipeline as ImbPipeline


### CONFIG -- the only section that changes per dataset/experiment

In [3]:
DATA_PATH      = "/content/drive/MyDrive/benchmark2/prepared_placement.csv"   # output of prepare_dataset_for_pipeline.py
TARGET_COL     = "program_stream"           # standardized target name
COLS_TO_REMOVE = []                          # any known junk columns still present
RANDOM_STATE   = 42

# NOTE: there is no fixed N_FEATURES_TO_KEEP here anymore. V4/V5/V6's
# "Top 15" / "Top 20" feature-selection targets are computed dynamically
# per dataset inside prepare_v2_style() as K15 = min(15, n_features) and
# K20 = min(20, n_features), and threaded through the `prep` dict. This
# is what makes ONE notebook work across datasets with very different
# predictor counts (e.g. 12 for benchmark2, hundreds for HSLS) without
# editing code -- only DATA_PATH/TARGET_COL/COLS_TO_REMOVE should change
# between datasets. For datasets with fewer than 15/20 predictors, K15
# and/or K20 simply equal the full feature count -- feature-selection
# reduction is not applicable there, and V4's output will show it
# (multiple configs converging on the same feature set), not fail.
# RFE_STEP=1 matches the individual V1-V6 notebooks exactly (remove one
# feature per elimination round). This is slower than a larger step once
# feature counts get large (e.g. 377 candidates like Set D) but is what
# the individual notebooks actually used, so it's the default here for
# benchmark parity. Bump it back up deliberately for large-feature runs
# where you're not trying to match a specific published notebook.
RFE_STEP = 1

# V5 tuning uses GridSearchCV (exhaustive grid search) to match the
# individual V1-V6 notebooks, not RandomizedSearchCV. No n_iter needed --
# GridSearchCV evaluates every combination in each model's param_grid.


### SHARED: load data once

In [4]:
def load_data(path: str, target_col: str, cols_to_remove: list) -> pd.DataFrame:
    df = pd.read_csv(path)
    present = [c for c in cols_to_remove if c in df.columns]
    if present:
        df = df.drop(columns=present)
    if df[target_col].isnull().sum() > 0:
        before = len(df)
        df = df.dropna(subset=[target_col]).reset_index(drop=True)
        print(f"Dropped {before - len(df)} rows with missing target.")
    print(f"Loaded: {df.shape[0]} rows x {df.shape[1]} columns")
    return df


def diagnose(f1_train, f1_test):
    gap = f1_train - f1_test
    if gap > 0.15:
        return gap, "OVERFIT"
    elif f1_test < 0.50 and f1_train < 0.50:
        return gap, "UNDERFIT"
    return gap, "GOOD FIT"


def evaluate(model, X_train, y_train, X_test, y_test, cv, name, extra=None):
    y_pred = model.predict(X_test)
    y_train_pred = model.predict(X_train)

    acc = accuracy_score(y_test, y_pred)
    prec = precision_score(y_test, y_pred, average="macro", zero_division=0)
    rec = recall_score(y_test, y_pred, average="macro", zero_division=0)
    f1_test = f1_score(y_test, y_pred, average="macro", zero_division=0)
    f1_train = f1_score(y_train, y_train_pred, average="macro", zero_division=0)
    gap, diag = diagnose(f1_train, f1_test)

    cv_scores = cross_val_score(model, X_train, y_train, cv=cv, scoring="f1_macro", n_jobs=-1)

    row = {
        "Model": name,
        "Accuracy": round(acc, 4),
        "Precision": round(prec, 4),
        "Recall": round(rec, 4),
        "F1 Test": round(f1_test, 4),
        "F1 Train": round(f1_train, 4),
        "Train-Test Gap": round(gap, 4),
        "CV Mean": round(cv_scores.mean(), 4),
        "CV Std": round(cv_scores.std(), 4),
        "Diagnosis": diag,
    }
    if extra:
        row.update(extra)
    return row


BASE_MODEL_DEFS = lambda: {
    "Decision Tree": DecisionTreeClassifier(random_state=RANDOM_STATE),
    "Random Forest": RandomForestClassifier(n_estimators=100, random_state=RANDOM_STATE, n_jobs=-1),
    "SVM": SVC(kernel="rbf", probability=True, random_state=RANDOM_STATE),
    "KNN": KNeighborsClassifier(n_neighbors=5),
    "Naive Bayes": GaussianNB(),
}


def smote_k_for(y_train):
    min_class_count = pd.Series(y_train).value_counts().min()
    return max(1, min(5, min_class_count - 1))


def make_cv(y):
    """StratifiedKFold with n_splits capped by the smallest class's size,
    not hardcoded to 5. A class with only 2-3 members (common in
    imbalanced multi-class data) can't support 5-fold CV; this adapts
    instead of silently erroring deep inside sklearn or, worse, running
    with folds that don't actually contain every class."""
    min_class_count = pd.Series(y).value_counts().min()
    n_splits = min(5, int(min_class_count))
    if n_splits < 2:
        raise ValueError(
            f"Cross-validation impossible: smallest class has only "
            f"{min_class_count} sample(s), need at least 2 for any CV split."
        )
    if n_splits < 5:
        print(f"  NOTE: smallest class has {min_class_count} samples -- "
              f"using {n_splits}-fold CV instead of the default 5-fold.")
    return StratifiedKFold(n_splits=n_splits, shuffle=True, random_state=RANDOM_STATE)


class ColumnIndexSelector(BaseEstimator, TransformerMixin):
    """Selects fixed column positions from an already-preprocessed array.
    Used by V4 to apply its one-time feature ranking (chi2/RFE/RF-
    importance, computed once -- exploratory, matches the original V4
    design and the benchmark papers' own methodology) inside a pipeline
    whose IMPUTATION/ENCODING/SCALING still gets refit fresh per CV
    fold. Only the selected feature *positions* are fixed in advance;
    the preprocessing that produces those positions' values is not."""
    def __init__(self, indices):
        self.indices = indices
    def fit(self, X, y=None):
        return self
    def transform(self, X):
        return X[:, self.indices]


def determine_column_types(df_train_raw: pd.DataFrame, target_col: str):
    """Classifies each predictor as continuous / discrete / categorical
    from the training schema. This is a one-time STRUCTURAL decision
    (which imputation strategy and encoder a column needs), not a
    fold-sensitive statistic -- same spirit as the nominal/ordinal
    variable audit, safe to do once outside CV."""
    numeric_cols = df_train_raw.select_dtypes(include="number").columns.tolist()
    categorical_cols = [c for c in df_train_raw.select_dtypes(include="object").columns if c != target_col]
    continuous_cols, discrete_cols = [], []
    for col in numeric_cols:
        if col == target_col:
            continue
        non_null = df_train_raw[col].dropna()
        is_integer = non_null.apply(lambda x: x == int(x)).all() if len(non_null) else True
        n_unique = df_train_raw[col].nunique()
        if is_integer and n_unique <= 20:
            discrete_cols.append(col)
        else:
            continuous_cols.append(col)
    return continuous_cols, discrete_cols, categorical_cols


def build_preprocessor(continuous_cols, discrete_cols, categorical_cols):
    """Returns an UNFITTED ColumnTransformer: mean-impute continuous,
    median-impute discrete, mode-impute + ordinal-encode categorical
    (unseen categories map to -1 via handle_unknown, replacing the old
    manual per-column LabelEncoder + unseen-value patch -- OrdinalEncoder
    is the sklearn-native, ColumnTransformer-safe equivalent).
    Scaling (MinMaxScaler) is added as a SEPARATE step wherever this is
    used, not bundled in here -- imblearn's Pipeline rejects a nested
    sklearn Pipeline as one of its own steps, so ColumnTransformer and
    MinMaxScaler must stay as two flat top-level steps, not one
    sub-Pipeline. Nothing is fit here -- fitting happens fresh each time
    this is cloned into a Pipeline and that Pipeline is fit, which is
    what makes this fold-safe: cross_val_score/GridSearchCV clone and
    refit the whole thing per fold, so imputation/encoding statistics
    never see a fold's held-out portion."""
    continuous_pipe = SkPipeline([("impute", SimpleImputer(strategy="mean"))])
    discrete_pipe = SkPipeline([("impute", SimpleImputer(strategy="median"))])
    categorical_pipe = SkPipeline([
        ("impute", SimpleImputer(strategy="most_frequent")),
        ("encode", OrdinalEncoder(handle_unknown="use_encoded_value", unknown_value=-1)),
    ])
    return ColumnTransformer(transformers=[
        ("cont", continuous_pipe, continuous_cols),
        ("disc", discrete_pipe, discrete_cols),
        ("cat", categorical_pipe, categorical_cols),
    ])


### V1 -- Raw baseline: dropna, label-encode, no scaling, no SMOTE

In [5]:
def run_v1(df: pd.DataFrame):
    print("\n" + "=" * 70)
    print("V1 -- Raw baseline (dropna, label-encode, no scaling, no SMOTE)")
    print("=" * 70)

    df_v1 = df.dropna().copy()
    print(f"Records after dropna: {len(df_v1)} (from {len(df)})")

    X_raw = df_v1.drop(columns=[TARGET_COL])
    y_raw = df_v1[TARGET_COL]

    X_train_raw, X_test_raw, y_train_labels, y_test_labels = train_test_split(
        X_raw, y_raw, test_size=0.2, random_state=RANDOM_STATE, stratify=y_raw
    )

    label_encoders = {}
    X_train = X_train_raw.copy()
    X_test = X_test_raw.copy()
    cat_cols = X_train_raw.select_dtypes(include="object").columns.tolist()

    for col in cat_cols:
        le = LabelEncoder()
        X_train[col] = le.fit_transform(X_train_raw[col].astype(str))
        known = set(le.classes_)
        test_vals = X_test_raw[col].astype(str)
        unseen = ~test_vals.isin(known)
        encoded_test = np.full(len(test_vals), -1, dtype=int)
        encoded_test[~unseen] = le.transform(test_vals[~unseen])
        X_test[col] = encoded_test
        label_encoders[col] = le

    le_target = LabelEncoder()
    le_target.fit(y_train_labels)
    y_train = le_target.transform(y_train_labels)
    y_test = le_target.transform(y_test_labels)

    cv = make_cv(y_train)
    results = []
    for name, model in BASE_MODEL_DEFS().items():
        model.fit(X_train, y_train)
        row = evaluate(model, X_train, y_train, X_test, y_test, cv, name)
        results.append(row)
        print(f"  {name:<15} F1 Test: {row['F1 Test']:.4f}  ({row['Diagnosis']})")

    return pd.DataFrame(results)


### SHARED: V2-V6 preprocessing (split before impute, then scale)

In [6]:
def prepare_v2_style(df: pd.DataFrame):
    """
    Split BEFORE any preprocessing (key structural fix vs V1). Unlike
    the previous version of this function, imputation/encoding/scaling
    are NOT fit here -- only column TYPES (continuous/discrete/
    categorical) are determined here, which is a structural decision,
    not a fold-sensitive statistic. Returns the RAW train/test feature
    frames plus a `make_preprocessor()` factory that builds a fresh,
    unfitted ColumnTransformer. Every V2-V6 pipeline embeds a clone of
    this preprocessor (+ MinMaxScaler) as its own first step, so
    imputation/encoding/scaling statistics get refit on only the
    training portion of each CV fold -- not on the full training split
    before CV/GridSearchCV ever sees it. This is the fix for the
    preprocessing-leakage issue: previously these statistics were fit
    once on the whole training split and the already-transformed array
    was handed to cross_val_score/GridSearchCV, so CV Mean and
    GridSearchCV's selection score were mildly optimistic. The held-out
    Test F1 was never affected either way, since the test set never
    participated in fitting those statistics under the old design.
    """
    target_missing = df[TARGET_COL].isnull().sum()
    if target_missing > 0:
        df = df.dropna(subset=[TARGET_COL]).reset_index(drop=True)

    train_idx, test_idx = train_test_split(
        df.index, test_size=0.2, random_state=RANDOM_STATE, stratify=df[TARGET_COL]
    )
    df_train_raw = df.loc[train_idx].copy()
    df_test_raw = df.loc[test_idx].copy()

    continuous_cols, discrete_cols, categorical_cols = determine_column_types(df_train_raw, TARGET_COL)
    feature_names = continuous_cols + discrete_cols + categorical_cols

    X_train_raw = df_train_raw[feature_names]
    X_test_raw = df_test_raw[feature_names]

    le_target = LabelEncoder()
    le_target.fit(df_train_raw[TARGET_COL])
    y_train = le_target.transform(df_train_raw[TARGET_COL])
    y_test = le_target.transform(df_test_raw[TARGET_COL])

    n_features = len(feature_names)
    K15 = min(15, n_features)
    K20 = min(20, n_features)
    print(f"Available predictors: {n_features} (continuous={len(continuous_cols)}, "
          f"discrete={len(discrete_cols)}, categorical={len(categorical_cols)})")
    print(f"V4 small feature set (K15): {K15}")
    print(f"V4/V5/V6 feature set (K20): {K20}")
    if K15 == n_features:
        print(f"  NOTE: K15 == all available predictors -- feature-selection "
              f"reduction is not applicable at the 15-feature threshold for "
              f"this dataset.")
    if K20 == n_features and K20 != K15:
        print(f"  NOTE: K20 == all available predictors -- feature-selection "
              f"reduction is not applicable at the 20-feature threshold for "
              f"this dataset.")

    def make_preprocessor():
        return build_preprocessor(continuous_cols, discrete_cols, categorical_cols)

    return {
        "X_train_raw": X_train_raw,
        "X_test_raw": X_test_raw,
        "y_train": y_train,
        "y_test": y_test,
        "feature_names": feature_names,
        "continuous_cols": continuous_cols,
        "discrete_cols": discrete_cols,
        "categorical_cols": categorical_cols,
        "make_preprocessor": make_preprocessor,
        "le_target": le_target,
        "n_features": n_features,
        "K15": K15,
        "K20": K20,
    }


### V2 -- Imputed + scaled, no SMOTE

In [7]:
def run_v2(prep: dict):
    print("\n" + "=" * 70)
    print("V2 -- Imputed + scaled (fold-safe), no SMOTE")
    print("=" * 70)

    X_train, X_test = prep["X_train_raw"], prep["X_test_raw"]
    y_train, y_test = prep["y_train"], prep["y_test"]
    cv = make_cv(y_train)

    results = []
    for name, model in BASE_MODEL_DEFS().items():
        pipe = SkPipeline([
            ("preprocess", prep["make_preprocessor"]()),
            ("scale", MinMaxScaler()),
            ("model", model),
        ])
        pipe.fit(X_train, y_train)
        row = evaluate(pipe, X_train, y_train, X_test, y_test, cv, name)
        results.append(row)
        print(f"  {name:<15} F1 Test: {row['F1 Test']:.4f}  ({row['Diagnosis']})")

    return pd.DataFrame(results)


### V3 -- V2 + SMOTE (nested inside the pipeline, refit per fold)

In [8]:
def run_v3(prep: dict):
    print("\n" + "=" * 70)
    print("V3 -- Imputed + scaled (fold-safe) + SMOTE")
    print("=" * 70)

    X_train, X_test = prep["X_train_raw"], prep["X_test_raw"]
    y_train, y_test = prep["y_train"], prep["y_test"]
    cv = make_cv(y_train)
    smote_k = smote_k_for(y_train)
    print(f"SMOTE k_neighbors={smote_k}")

    results = []
    for name, model in BASE_MODEL_DEFS().items():
        pipe = ImbPipeline([
            ("preprocess", prep["make_preprocessor"]()),
            ("scale", MinMaxScaler()),
            ("smote", SMOTE(random_state=RANDOM_STATE, k_neighbors=smote_k)),
            ("model", model),
        ])
        pipe.fit(X_train, y_train)
        row = evaluate(pipe, X_train, y_train, X_test, y_test, cv, name)
        results.append(row)
        print(f"  {name:<15} F1 Test: {row['F1 Test']:.4f}  ({row['Diagnosis']})")

    return pd.DataFrame(results)


### V4 -- V2 + feature selection (Chi2 / RFE / RF-importance), 8 configs

In [9]:
def run_v4(prep: dict):
    print("\n" + "=" * 70)
    print("V4 -- Feature selection (Chi2 / RFE / RF importance)")
    print("=" * 70)

    X_train_raw, X_test_raw = prep["X_train_raw"], prep["X_test_raw"]
    y_train, y_test = prep["y_train"], prep["y_test"]
    feature_names = prep["feature_names"]
    K15, K20 = prep["K15"], prep["K20"]
    cv = make_cv(y_train)
    smote_k = smote_k_for(y_train)

    if K15 == K20:
        # K15 == min(15, n) and K20 == min(20, n) can only be equal when
        # n <= 15 -- i.e. this dataset has too few predictors for the
        # 15-vs-20 comparison to mean anything. Without this guard, all
        # 8 configs below would select the SAME n features (just via
        # different ranking methods, in different orders), collide as
        # dict keys where methods happen to produce identical label text
        # (e.g. two configs both named "Chi-Square Top 12"), silently
        # collapsing 8 intended configs down to fewer, and the surviving
        # "duplicates" would show different F1 purely from column-order
        # sensitivity in downstream models (esp. tree splits, distance
        # metrics) -- not from any real feature-selection effect. Report
        # this honestly as one config instead of manufacturing a false
        # comparison, and skip the ranking computation entirely since
        # its output wouldn't be used for anything.
        print(f"NOTE: K15 == K20 == {K15} (all available predictors) -- "
              f"feature-selection reduction is not applicable for this "
              f"dataset. Reporting a single 'all features' configuration "
              f"instead of 8 reordered copies of the same feature set.")
        feature_configs = {
            "All features (selection not applicable)": feature_names,
        }
        v4_feature_sets = {"selection_not_applicable": True, "all_features": feature_names}
    else:
        # One-time preprocessing fit on the FULL training split, used ONLY
        # to RANK features for choosing which columns go into each config
        # below. This matches the original V4 methodology and the
        # benchmark papers' own approach: feature ranking is a one-time
        # exploratory step, not something that needs to be refit per
        # fold. What DOES get refit per fold is the actual model SCORING
        # further down, via a fresh preprocessor inside each config's
        # pipeline.
        exploratory_pre = SkPipeline([
            ("preprocess", prep["make_preprocessor"]()),
            ("scale", MinMaxScaler()),
        ])
        X_train_for_ranking = exploratory_pre.fit_transform(X_train_raw)

        # Method 1: Chi-square (requires non-negative features -- MinMax-scaled, so OK)
        chi2_selector = SelectKBest(score_func=chi2, k="all")
        chi2_selector.fit(X_train_for_ranking, y_train)
        chi2_ranked = pd.Series(chi2_selector.scores_, index=feature_names).sort_values(ascending=False)
        chi2_top15 = list(chi2_ranked.head(K15).index)
        chi2_top20 = list(chi2_ranked.head(K20).index)

        # Method 2: RFE with Random Forest
        rf_for_rfe = RandomForestClassifier(n_estimators=50, random_state=RANDOM_STATE, n_jobs=-1)
        rfe15 = RFE(estimator=rf_for_rfe, n_features_to_select=K15, step=RFE_STEP).fit(X_train_for_ranking, y_train)
        rfe_top15 = list(pd.Series(rfe15.ranking_, index=feature_names).pipe(lambda s: s[s == 1].index))
        rfe20 = RFE(estimator=rf_for_rfe, n_features_to_select=K20, step=RFE_STEP).fit(X_train_for_ranking, y_train)
        rfe_top20 = list(pd.Series(rfe20.ranking_, index=feature_names).pipe(lambda s: s[s == 1].index))

        # Method 3: RF importance
        rf_importance = RandomForestClassifier(n_estimators=100, random_state=RANDOM_STATE, n_jobs=-1)
        rf_importance.fit(X_train_for_ranking, y_train)
        importances_ranked = pd.Series(rf_importance.feature_importances_, index=feature_names).sort_values(ascending=False)
        rf_top15 = list(importances_ranked.head(K15).index)
        rf_top20 = list(importances_ranked.head(K20).index)

        in_all_three = set(chi2_top15) & set(rfe_top15) & set(rf_top15)

        feature_configs = {
            "All features (V3 baseline)": feature_names,
            f"Chi-Square Top {K15}": chi2_top15,
            f"Chi-Square Top {K20}": chi2_top20,
            f"RFE Top {K15}": rfe_top15,
            f"RFE Top {K20}": rfe_top20,
            f"RF Importance Top {K15}": rf_top15,
            f"RF Importance Top {K20}": rf_top20,
            "Consensus (all 3 methods)": sorted(list(in_all_three)),
        }
        v4_feature_sets = {
            "chi2_top20": chi2_top20, "rfe_top20": rfe_top20, "rf_top20": rf_top20,
            "consensus": sorted(list(in_all_three)),
        }

    print(f"Feature configurations to test ({len(feature_configs)}):")
    for name, feats in feature_configs.items():
        print(f"  {name:<30} {len(feats)} features")

    results = []
    for config_name, selected_features in feature_configs.items():
        if len(selected_features) == 0:
            continue
        idx = [feature_names.index(f) for f in selected_features if f in feature_names]

        for model_name, base_model in BASE_MODEL_DEFS().items():
            pipe = ImbPipeline([
                ("preprocess", prep["make_preprocessor"]()),
                ("scale", MinMaxScaler()),
                ("select", ColumnIndexSelector(idx)),
                ("smote", SMOTE(random_state=RANDOM_STATE, k_neighbors=smote_k)),
                ("clf", base_model),
            ])
            pipe.fit(X_train_raw, y_train)
            row = evaluate(pipe, X_train_raw, y_train, X_test_raw, y_test, cv, model_name,
                            extra={"Feature Config": config_name, "N Features": len(idx)})
            results.append(row)

        best_this_config = max(
            (r for r in results if r["Feature Config"] == config_name),
            key=lambda r: r["F1 Test"],
        )
        print(f"  Best in '{config_name}': {best_this_config['Model']} "
              f"(F1 Test={best_this_config['F1 Test']:.4f})")

    results_df = pd.DataFrame(results)
    return results_df, v4_feature_sets


### V5 -- RFE-Top-20 (V4 winner) + SMOTE + GridSearchCV tuning

In [10]:
def run_v5(prep: dict):
    print("\n" + "=" * 70)
    print(f"V5 -- RFE Top {prep['K20']} + SMOTE + GridSearchCV tuning")
    print("=" * 70)

    X_train, X_test = prep["X_train_raw"], prep["X_test_raw"]
    y_train, y_test = prep["y_train"], prep["y_test"]
    cv = make_cv(y_train)
    smote_k = smote_k_for(y_train)

    def make_selector():
        return RFE(
            estimator=RandomForestClassifier(n_estimators=50, random_state=RANDOM_STATE, n_jobs=-1),
            n_features_to_select=prep["K20"], step=RFE_STEP,
        )

    base_models = BASE_MODEL_DEFS()
    pipelines = {
        name: ImbPipeline([
            ("preprocess", prep["make_preprocessor"]()),
            ("scale", MinMaxScaler()),
            ("select", make_selector()),
            ("smote", SMOTE(random_state=RANDOM_STATE, k_neighbors=smote_k)),
            ("model", model),
        ])
        for name, model in base_models.items()
    }

    param_grids = {
        "Decision Tree": {
            "model__max_depth": [3, 5, 7, 10, None],
            "model__min_samples_leaf": [1, 2, 5, 10],
            "model__criterion": ["gini", "entropy"],
        },
        "Random Forest": {
            "model__n_estimators": [100, 200, 300],
            "model__max_depth": [5, 10, 20, None],
            "model__min_samples_leaf": [1, 2, 5],
        },
        "SVM": {
            "model__C": [0.1, 1, 10, 100],
            "model__gamma": ["scale", "auto", 0.01, 0.1],
        },
        "KNN": {
            "model__n_neighbors": [3, 5, 7, 9, 11, 15],
            "model__weights": ["uniform", "distance"],
        },
        "Naive Bayes": {
            "model__var_smoothing": np.logspace(0, -9, 10),
        },
    }

    results = []
    tuned_models = {}
    for model_name, pipeline in pipelines.items():
        print(f"\nTuning: {model_name}")
        # GridSearchCV -- exhaustive search over every combination in
        # param_grids[model_name], matching the individual V5 notebook.
        # Slower than RandomizedSearchCV once feature counts / grids get
        # large, but this is what the individual notebook actually ran.
        grid = GridSearchCV(
            estimator=pipeline, param_grid=param_grids[model_name],
            cv=cv, scoring="f1_macro",
            n_jobs=-1, refit=True,
        )
        grid.fit(X_train, y_train)
        best = grid.best_estimator_
        tuned_models[model_name] = best
        print(f"  Best params: {grid.best_params_}")

        row = evaluate(best, X_train, y_train, X_test, y_test, cv, model_name,
                        extra={"Best Params": str(grid.best_params_)})
        results.append(row)
        print(f"  F1 Test: {row['F1 Test']:.4f}  ({row['Diagnosis']})")

    return pd.DataFrame(results), tuned_models


### V6 -- Stacking ensemble: RFE-Top-20 + SMOTE + Stacking(RF+SVM+NB->LogReg)

In [11]:
def run_v6(prep: dict, v5_tuned_models: dict = None):
    print("\n" + "=" * 70)
    print("V6 -- Stacking ensemble (RF + SVM + NB -> Logistic Regression)")
    print("=" * 70)

    X_train, X_test = prep["X_train_raw"], prep["X_test_raw"]
    y_train, y_test = prep["y_train"], prep["y_test"]
    cv = make_cv(y_train)
    smote_k = smote_k_for(y_train)

    def make_selector():
        return RFE(
            estimator=RandomForestClassifier(n_estimators=50, random_state=RANDOM_STATE, n_jobs=-1),
            n_features_to_select=prep["K20"], step=RFE_STEP,
        )

    # DYNAMIC handoff: base learners use THIS dataset's V5 tuned
    # hyperparameters, via clone() of the fitted pipeline's model step --
    # not hardcoded constants from a different dataset/run. This is what
    # makes V6 correct across datasets: Dataset A's V5 tunes Dataset A,
    # V6 uses Dataset A's params; swap in Dataset B and V6 automatically
    # uses Dataset B's params, with no code edit required.
    def get_tuned(name, fallback):
        if v5_tuned_models and name in v5_tuned_models:
            try:
                return clone(v5_tuned_models[name].named_steps["model"])
            except Exception as e:
                print(f"  WARNING: could not clone tuned {name} ({e}), using fallback default.")
        else:
            print(f"  NOTE: no V5 tuned model found for {name} -- using fallback default "
                  f"(V5 may have failed, or wasn't run first).")
        return fallback

    rf = get_tuned("Random Forest", RandomForestClassifier(
        n_estimators=100, random_state=RANDOM_STATE, n_jobs=-1))
    svm = get_tuned("SVM", SVC(kernel="rbf", probability=True, random_state=RANDOM_STATE))
    nb = get_tuned("Naive Bayes", GaussianNB())

    base_estimators = [("rf", rf), ("svm", svm), ("nb", nb)]
    print("Base learners (tuned params from this dataset's V5, where available):")
    for name, est in base_estimators:
        print(f"  {name}: {est}")

    stacking_model = StackingClassifier(
        estimators=base_estimators,
        final_estimator=LogisticRegression(max_iter=1000, random_state=RANDOM_STATE),
        cv=cv, n_jobs=-1,
    )

    pipeline_v6 = ImbPipeline([
        ("preprocess", prep["make_preprocessor"]()),
        ("scale", MinMaxScaler()),
        ("select", make_selector()),
        ("smote", SMOTE(random_state=RANDOM_STATE, k_neighbors=smote_k)),
        ("model", stacking_model),
    ])

    pipeline_v6.fit(X_train, y_train)
    row = evaluate(pipeline_v6, X_train, y_train, X_test, y_test, cv,
                    "Stacking (RF+SVM+NB -> LogReg)")
    print(f"  F1 Test: {row['F1 Test']:.4f}  ({row['Diagnosis']})")

    return pd.DataFrame([row])


### MAIN -- run all six versions and print the final comparison

In [12]:
def main():
    df = load_data(DATA_PATH, TARGET_COL, COLS_TO_REMOVE)

    results = {}
    summary_rows = []

    def safe_run(label, fn, *args):
        try:
            out = fn(*args)
            return out, None
        except Exception as e:
            print(f"\n*** {label} FAILED: {e} ***")
            print(f"*** This is reported, not hidden -- see summary at the end. ***")
            return None, str(e)

    v1_df, v1_err = safe_run("V1", run_v1, df)
    if v1_df is not None:
        results["v1"] = v1_df

    prep = prepare_v2_style(df)  # V2-V6 share this; let it raise loudly if it fails

    v2_df, v2_err = safe_run("V2", run_v2, prep)
    if v2_df is not None:
        results["v2"] = v2_df

    v3_df, v3_err = safe_run("V3", run_v3, prep)
    if v3_df is not None:
        results["v3"] = v3_df

    v4_out, v4_err = safe_run("V4", run_v4, prep)
    if v4_out is not None:
        v4_df, v4_feature_sets = v4_out
        results["v4"] = v4_df
    else:
        v4_df = None

    v5_out, v5_err = safe_run("V5", run_v5, prep)
    if v5_out is not None:
        v5_df, v5_tuned_models = v5_out
        results["v5"] = v5_df
    else:
        v5_df, v5_tuned_models = None, None

    v6_df, v6_err = safe_run("V6", run_v6, prep, v5_tuned_models)
    if v6_df is not None:
        results["v6"] = v6_df

    def best_row(df_results, version_label, err):
        if df_results is None:
            return {"Version": version_label, "Best Model": "FAILED", "F1 Test": None, "Note": err}
        best = df_results.loc[df_results["F1 Test"].idxmax()]
        return {"Version": version_label, "Best Model": best["Model"], "F1 Test": best["F1 Test"], "Note": ""}

    summary = pd.DataFrame([
        best_row(v1_df, "V1 (raw baseline)", v1_err),
        best_row(results.get("v2"), "V2 (imputed + scaled)", v2_err),
        best_row(results.get("v3"), "V3 (+ SMOTE)", v3_err),
        best_row(v4_df, "V4 (+ feature selection)", v4_err),
        best_row(v5_df, "V5 (+ hyperparameter tuning)", v5_err),
        best_row(results.get("v6"), "V6 (stacking ensemble)", v6_err),
    ])

    print("\n" + "=" * 70)
    print("FINAL V1 -> V6 COMPARISON (best model per version, by F1 Test)")
    print("=" * 70)
    print(summary.to_string(index=False))

    summary.to_csv("all_versions_summary.csv", index=False)
    if v4_df is not None:
        v4_df.to_csv("v4_all_feature_configs_results.csv", index=False)
        print("\nSaved: all_versions_summary.csv, v4_all_feature_configs_results.csv")
    else:
        print("\nSaved: all_versions_summary.csv")

    results["summary"] = summary
    return results


## Run all six versions

In [13]:
results = main()


Loaded: 215 rows x 13 columns

V1 -- Raw baseline (dropna, label-encode, no scaling, no SMOTE)
Records after dropna: 215 (from 215)
  Decision Tree   F1 Test: 0.6742  (OVERFIT)
  Random Forest   F1 Test: 0.7350  (OVERFIT)
  SVM             F1 Test: 0.6595  (GOOD FIT)
  KNN             F1 Test: 0.6512  (GOOD FIT)
  Naive Bayes     F1 Test: 0.6917  (GOOD FIT)
Available predictors: 12 (continuous=6, discrete=0, categorical=6)
V4 small feature set (K15): 12
V4/V5/V6 feature set (K20): 12
  NOTE: K15 == all available predictors -- feature-selection reduction is not applicable at the 15-feature threshold for this dataset.

V2 -- Imputed + scaled (fold-safe), no SMOTE
  Decision Tree   F1 Test: 0.6277  (OVERFIT)
  Random Forest   F1 Test: 0.7611  (OVERFIT)
  SVM             F1 Test: 0.5713  (OVERFIT)
  KNN             F1 Test: 0.5905  (OVERFIT)
  Naive Bayes     F1 Test: 0.6917  (GOOD FIT)

V3 -- Imputed + scaled (fold-safe) + SMOTE
SMOTE k_neighbors=5
  Decision Tree   F1 Test: 0.5793  (OVER